In [37]:
import os
import dotenv
from neo4j import GraphDatabase
import pandas as pd
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pyvis.network import Network
from IPython.display import IFrame

In [38]:

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))

driver = GraphDatabase.driver(URI, auth=AUTH)

DIAG_LABEL = "Diagnostico"   
DIAG_NAME_PROP = "terminoEN"     
PACIENTE_LABEL = "Paciente"      

def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

In [39]:
# 1. Comprobar número de pacientes y cargar nombres de diagnósticos
q_count_pat = f"MATCH (p:{PACIENTE_LABEL}) RETURN count(p) AS cnt"
cnt_pat = run_query(q_count_pat)[0]["cnt"]
print(f"Pacientes encontrados: {cnt_pat}")

q_nodos = f"MATCH (d:{DIAG_LABEL}) RETURN elementId(d) AS id, d.{DIAG_NAME_PROP} AS nombre"
nodos_raw = run_query(q_nodos)
id2name = {row["id"]: row["nombre"] or f"diag_{row['id']}" for row in nodos_raw}
print(f"Diagnósticos cargados: {len(id2name)}")

Pacientes encontrados: 1567
Diagnósticos cargados: 1746


In [40]:

# 2. Construir el grafo de co-ocurrencia no dirigido
G = nx.Graph()

for nid, nombre in id2name.items():
    G.add_node(nid, label=nombre)

if cnt_pat > 0:
    q_rels = f"""
    MATCH (p:{PACIENTE_LABEL})-[r]->(d:{DIAG_LABEL})
    RETURN elementId(p) AS pid, collect(DISTINCT elementId(d)) AS diagnosticos
    """
    rels = run_query(q_rels)
    for row in rels:
        diags = row["diagnosticos"] or []
        for i in range(len(diags)):
            for j in range(i + 1, len(diags)):
                a, b = diags[i], diags[j]
                if G.has_edge(a, b):
                    G[a][b]["weight"] += 1
                else:
                    G.add_edge(a, b, weight=1)
    print("Grafo de co-ocurrencia construido a partir de pacientes.")
else:
    q_diag_links = f"""
    MATCH (d1:{DIAG_LABEL})-[r]->(d2:{DIAG_LABEL})
    RETURN elementId(d1) AS a, elementId(d2) AS b, type(r) AS relType, count(*) AS cnt
    """
    links = run_query(q_diag_links)
    for row in links:
        a, b, cnt = row["a"], row["b"], row["cnt"]
        if G.has_edge(a, b):
            G[a][b]["weight"] += cnt
        else:
            G.add_edge(a, b, weight=cnt)

print(f"➜ Grafo original cargado: {G.number_of_nodes()} nodos | {G.number_of_edges()} aristas")

Grafo de co-ocurrencia construido a partir de pacientes.
➜ Grafo original cargado: 1746 nodos | 40657 aristas


In [41]:
# 1. Analizar los pesos encontrados
pesos = [d.get("weight", 1) for u, v, d in G.edges(data=True)]
if pesos:
    print(f"➜ Peso máximo hallado: {max(pesos)}")
    print(f"➜ Peso promedio: {sum(pesos)/len(pesos):.2f}")

# 2. Aplicar filtro por umbral (ajusta a 5, 8 o 10 según la densidad deseada)
MIN_PESO = 3

aristas_a_eliminar = [
    (u, v) for u, v, d in G.edges(data=True) if d.get("weight", 0) < MIN_PESO
]
G.remove_edges_from(aristas_a_eliminar)

# 3. Eliminar nodos que quedaron aislados
G.remove_nodes_from(list(nx.isolates(G)))

print(f"➜ Grafo filtrado (MIN_PESO >= {MIN_PESO}): {G.number_of_nodes()} nodos | {G.number_of_edges()} aristas")

➜ Peso máximo hallado: 419
➜ Peso promedio: 1.80
➜ Grafo filtrado (MIN_PESO >= 3): 482 nodos | 4391 aristas


In [42]:
# Computar partición de Louvain ponderada por el peso de la relación
partition = community_louvain.best_partition(G, weight="weight")

# Guardar DataFrame para análisis de la investigación
louvain_df = pd.DataFrame([
    {
        "node_id": node,
        "diagnostico": G.nodes[node].get("label", id2name.get(node, str(node))),
        "comunidad": community,
        "degree_ponderado": G.degree(node, weight="weight")
    }
    for node, community in partition.items()
])

num_comunidades = louvain_df["comunidad"].nunique()
print(f"Número de comunidades detectadas en la red filtrada: {num_comunidades}")
louvain_df.sort_values(["comunidad", "degree_ponderado"], ascending=[True, False]).head(15)

Número de comunidades detectadas en la red filtrada: 5


,node_id,diagnostico,comunidad,degree_ponderado
130,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:371,"Esquizofrenia, no especificada",0,3320
122,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:363,Esquizofrenia,0,2978
117,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:350,"Dependencia de nicotina no especificada, sin c...",0,1566
127,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:368,Esquizofrenia residual,0,525
267,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:842,Hernia diafragmática sin obstrucción ni gangrena,0,414
292,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:913,Cálculo de vesícula biliar sin colecistitis si...,0,251
91,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:284,Hiposmolaridad e hiponatremia,0,188
454,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:1685,Ausencia adquirida de otras partes especificad...,0,171
37,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:114,Neoplasia maligna secundaria de hígado y vías ...,0,140
389,4:90c38f6b-b92e-449f-835e-bd1b95aa4168:1542,Contacto para cuidados paliativos,0,110


In [43]:
# Agrupar por comunidad y extraer los diagnósticos más importantes
comunidades_resumen = (
    louvain_df.groupby("comunidad")
    .agg(
        num_diagnosticos=("diagnostico", "count"),
        diagnosticos_clave=("diagnostico", lambda x: ", ".join(x.head(4)))
    )
    .sort_values("num_diagnosticos", ascending=False)
)

print("--- Top Comunidades de Diagnósticos ---")
display(comunidades_resumen.head(10))

--- Top Comunidades de Diagnósticos ---


,num_diagnosticos,diagnosticos_clave
comunidad,,
0,158,"Gastroenteritis y colitis infecciosas, no espe..."
3,144,"Hepatitis vírica crónica tipo C, Secuelas de t..."
4,98,"Sepsis por Escherichia coli [E. coli], Infecci..."
1,80,"Sepsis, microorganismo no especificado, Estoma..."
2,2,Diabetes mellitus tipo 1 con polineuropatía di...


In [44]:
# Paleta de colores para diferenciar cada comunidad
cmap = plt.get_cmap("tab20", max(num_comunidades, 1))

def get_community_color(comm_id):
    rgb = cmap(comm_id % 20)[:3]
    return mcolors.to_hex(rgb)

net = Network(height="800px", width="100%", bgcolor="#ffffff", font_color="#111111")

# Añadir nodos configurados con su nombre real y su color de comunidad
for node in G.nodes():
    label = G.nodes[node].get("label", str(node))
    comm = partition.get(node, 0)
    deg = G.degree(node)
    
    short_label = label[:25] + "..." if len(label) > 25 else label

    net.add_node(
        node,
        label=short_label,
        title=f"<b>{label}</b><br>Comunidad: {comm}<br>Grado (conexiones): {deg}",
        value=deg * 3 + 4,
        color=get_community_color(comm)
    )

# Añadir aristas
for source, target, data in G.edges(data=True):
    w = data.get("weight", 1)
    net.add_edge(source, target, value=w, color="#e0e0e0")

# Opciones de física para separar comunidades y evitar colisiones de texto
net.set_options("""
var options = {
  "nodes": {
    "font": {
      "size": 13,
      "face": "arial",
      "strokeWidth": 3,
      "strokeColor": "#ffffff"
    }
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -70,
      "centralGravity": 0.01,
      "springLength": 130,
      "springConstant": 0.08,
      "avoidOverlap": 1
    },
    "maxVelocity": 50,
    "solver": "forceAtlas2Based",
    "timestep": 0.35,
    "stabilization": { "iterations": 200 }
  }
}
""")


html_path = "louvain_network.html"
net.write_html(html_path, open_browser=False)
display(IFrame(html_path, width="100%", height="800px"))

In [45]:
driver.close()